# Single-path trajectory analog for ReLU nets: frozen sign-pattern cell

The ReLU counterpart of the single-path DLGN trajectory study
(*Layers as Lenses*, Sec. 3.3, Fig. "parameter and gradient trajectories"), built so
that the **hierarchy-seeking lens theorem** (Thm. 1) and the **self-opposing lens
theorem** (Thm. 2) have a direct empirical analog.

## The model

Depth-$D$ ODT labels, $d$-dimensional data on the sphere, ReLU net with **no biases**:

$$\hat y(x) \;=\; \sum_{s\in\{\pm1\}^D} a_s\,\phi\!\Big(\sum_{k=1}^{D} s_k\,\phi(w_k^\top x)\Big),
\qquad \phi = \mathrm{ReLU}.$$

- **Layer 1** — $D$ neurons $w_1,\dots,w_D \in \mathbb{R}^d$. *Trainable*, hand-initialised.
- **Layer 2** — $2^D$ neurons, row $s$ of $W_2$ is a **unique $\{+1,-1\}^D$ bit vector**. *Frozen.*
- **Output** — $a \in \{+1,-1\}^{2^D}$, random. *Frozen.*

Only $W_1$ moves, under **population gradient flow** (full-batch GD, constant step size).

## Why this is the right analog

In the single-path DLGN, $\hat y = a\prod_\ell \phi(w_\ell^\top x)$: the $D$ gates enter
multiplicatively, and the lens on $w_k$ is the product of the others. Here the frozen
second layer enumerates **every sign pattern** of the $D$ first-layer gates, which is how
a ReLU net forms conjunctions of its gates. The gradient keeps the same three factors:

$$-\nabla_{w_k}\mathcal{L}
= \mathbb{E}\Big[\underbrace{\sigma(-y\hat y)}_{\text{Residue}}\;
\underbrace{\Lambda_k(x)}_{\text{Lens}}\;
\underbrace{\mathbb{1}[w_k^\top x>0]}_{\text{Blinder}}\; y\,x\Big],
\qquad
\Lambda_k(x)=\sum_{s} a_s\,s_k\,\mathbb{1}\Big[\textstyle\sum_j s_j \phi(w_j^\top x)>0\Big].$$

$\Lambda_k$ is built from *all* first-layer neurons through the layer-2 gates, so each
$w_k$ is again both a separator (on lens-weighted data) and a component of everyone
else's lens. That is the positive feedback loop, now with the lens **additive over sign
patterns** instead of a single product. Because layers 2 and 3 are frozen, the loop runs
entirely within the first layer — no confound from downstream weights adapting.

## One ReLU-specific caveat

The paper's construction starts from $W=0$, which is fine for DLGN ($\phi$ is a soft gate,
$\phi(0)=\tfrac12$). For ReLU, $W_1 = 0$ gives $h_1 \equiv 0$, every layer-2 pre-activation
is $0$, and the gradient vanishes identically — an exactly degenerate stationary point.
So the init below is *paper perturbation + a small component orthogonal to every ODT
direction*, which is also the regime the theorems assume ($w_\lambda$ nearly orthogonal
to all $u_i$, $d \gg D$). `INIT_RANDOM_SCALE` controls that component.

In [1]:
from itertools import product
from pathlib import Path
import pickle
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

%load_ext autoreload
%autoreload 2

from layer_lenses.odt import generate_cob_odt_data, odt_leaf_ids_for_x
from layer_lenses.relu_analysis import (
    activation_tensor_by_layer_neuron_data,
    collect_start_end_metrics,
    first_layer_neuron_weighted_lens,
    first_layer_odt_alignment,
    log_loss_gradients,
)
from layer_lenses.relu_mlp import ReLUMLP
from layer_lenses.relu_training import (
    ReLUTrainConfig,
    checkpoint_model_from_state,
    train_relu_mlp,
)
from layer_lenses.utilities import plot_complete_binary_tree_values

print("cuda available:", torch.cuda.is_available())

cuda available: True


## 1. Configuration

Gradient-flow settings: full batch (population gradient), plain SGD with no momentum and
no weight decay, and `lr_scheduler_eta_min_ratio=1.0` so the step size stays **constant**
(the cosine schedule would otherwise decay it). `LR` is the flow step size.

In [2]:
# --- problem ---------------------------------------------------------------
DIM = 100
DEPTH = 4                      # D: ODT depth = number of first-layer neurons
N_TRAIN = 200_000              # population sample for the gradient
DATA_SEED = 178                # paper's trajectory figure used seed tag 0178

# --- architecture ----------------------------------------------------------
WIDTH_1 = DEPTH                # D first-layer neurons (trainable)
WIDTH_2 = 2**DEPTH             # one frozen neuron per sign pattern
OUTPUT_SIGN_SEED = 178         # seed for the frozen random +-1 output vector

# --- initialisation --------------------------------------------------------
EPS = 0.1                      # paper's epsilon in the Sec. 3.3 perturbation
INIT_RANDOM_SCALE = 0.02       # size of the off-ODT ("spurious") init component
INIT_RANDOM_SEED = 178

# --- gradient flow ---------------------------------------------------------
TOTAL_EPOCHS = 2_000
LR = 0.5
TRAIN_SEED = 178

# --- analysis --------------------------------------------------------------
COS_THRESHOLD = 0.85
GRAD_TRAJ_MAX_CHECKPOINTS = 150   # checkpoints used for the gradient-trajectory figure
PROBE_N = 100_000                 # per split-half probe sample (needs 2 * PROBE_N <= N_TRAIN)
PROBE_EPS = 1e-2                  # finite-difference step; smaller values are noise-dominated,
                                  # see the split-half stability check in Sec. 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SNAPSHOT_EPOCHS = tuple(sorted(set(range(0, min(500, TOTAL_EPOCHS) + 1))
                               | set(range(0, TOTAL_EPOCHS + 1, 10))))

NUM_INTERNAL_NODES = 2**DEPTH - 1
NUM_LEAVES = 2**DEPTH
NODE_LEVEL = np.floor(np.log2(np.arange(NUM_INTERNAL_NODES) + 1)).astype(int)
DEEP_NODES = [i for i in range(NUM_INTERNAL_NODES) if NODE_LEVEL[i] == DEPTH - 1]

print(f"architecture: {DIM} -> {WIDTH_1} (trainable) -> {WIDTH_2} (frozen signs) -> 1 (frozen +-1)")
print(f"ODT: depth {DEPTH}, internal nodes 0..{NUM_INTERNAL_NODES - 1}, "
      f"deepest-level nodes {DEEP_NODES}")
print(f"{len(SNAPSHOT_EPOCHS)} snapshots, device={DEVICE}")

architecture: 100 -> 4 (trainable) -> 16 (frozen signs) -> 1 (frozen +-1)
ODT: depth 4, internal nodes 0..14, deepest-level nodes [7, 8, 9, 10, 11, 12, 13, 14]
651 snapshots, device=cuda


## 2. Data and ODT directions

The ODT normals $u_0,\dots,u_{14}$ are orthonormal, so $w\cdot u_i$ are literally the
coordinates of $w$ in the ODT basis — the quantity plotted in the paper's trajectory
figure.

In [3]:
x_all, y_all, tree, meta = generate_cob_odt_data(
    num_data=2 * N_TRAIN, dim=DIM, depth=DEPTH, seed=DATA_SEED
)
x_train, y_train = x_all[:N_TRAIN], y_all[:N_TRAIN]
x_eval, y_eval = x_all[N_TRAIN:], y_all[N_TRAIN:]

U = tree.w_list                                  # (num_internal_nodes, d)
gram = U @ U.T
assert np.allclose(gram, np.eye(U.shape[0]), atol=1e-8), "ODT normals are not orthonormal"

leaf_ids = odt_leaf_ids_for_x(x_train, tree)
leaf_offsets = leaf_ids - NUM_INTERNAL_NODES

print("x_train:", x_train.shape, " positive fraction:", float((y_train == 1).mean()))
print("ODT normals:", U.shape, "(orthonormal)")
meta

x_train: (200000, 100)  positive fraction: 0.499965
ODT normals: (15, 100) (orthonormal)


{'num_requested': 400000,
 'num_kept': 400000,
 'num_pruned': 0,
 'dim': 100,
 'depth': 4,
 'num_internal_nodes': 15,
 'num_leaf_nodes': 16,
 'threshold': 0.0,
 'seed': 178}

## 3. The frozen layers

Layer 2 rows enumerate $\{+1,-1\}^D$ in a fixed order; the label `"+--+"` reads off the
sign each first-layer neuron gets in that unit. The output vector is a frozen random
$\pm1$ sign per unit (set `OUTPUT_SIGNS` by hand to override).

In [ ]:
SIGN_PATTERNS = np.array(
    list(product([1.0, -1.0], repeat=WIDTH_1)), dtype=np.float64
)                                                       # (2^D, D)
PATTERN_LABELS = ["".join("+" if v > 0 else "-" for v in row) for row in SIGN_PATTERNS]

OUTPUT_SIGN_PRESETS = {
    # as specified: a random frozen +-1 per unit
    "random": np.random.default_rng(OUTPUT_SIGN_SEED).choice([-1.0, 1.0], size=WIDTH_2),
    # closest analog of the paper's fixed a = 1 > 0
    "all_positive": np.ones(WIDTH_2),
    # +1 when the pattern has at least as many + as - entries
    "majority": np.where(SIGN_PATTERNS.sum(axis=1) >= 0, 1.0, -1.0),
    "parity": SIGN_PATTERNS.prod(axis=1),
}

OUTPUT_SIGN_CHOICE = "random"
OUTPUT_SIGNS = np.asarray(
    OUTPUT_SIGN_PRESETS[OUTPUT_SIGN_CHOICE], dtype=np.float64
).reshape(WIDTH_2)

frozen_layers_df = pd.DataFrame(
    {
        "unit": range(WIDTH_2),
        "pattern": PATTERN_LABELS,
        "a_s": OUTPUT_SIGNS,
        "sum_s": SIGN_PATTERNS.sum(axis=1),
    }
)
frozen_layers_df

## 4. Hand-picked first-layer initialisation

**This is the cell to edit.** `init_coeffs[k, i]` is the coefficient of $u_i$ in
$w_k(0)$; everything else is derived. The default reproduces the paper's Sec. 3.3
perturbation for $D=4$,

$$w_1 = 0,\quad w_2 = -\epsilon v,\quad w_3 = -\epsilon v - \epsilon u_7,\quad
w_4 = -\epsilon v - \epsilon u_8, \qquad v = u_7+u_8+u_9+u_{10},$$

plus the small off-ODT component discussed above. From $w_1$'s viewpoint the other
neurons have moved along $-u_7,\dots,-u_{10}$, so Thm. 1 predicts an increased push on
$w_1$ toward $-u_0$; the extra $-\epsilon u_7$, $-\epsilon u_8$ on $w_3,w_4$ should
similarly push $w_2$ toward $-u_1$. The DLGN run converged to
$\upsilon = [0, 1, 7, 8]$.

In [ ]:
V_NODES = [7, 8, 9, 10]     # paper's v = u_7 + u_8 + u_9 + u_10

# Row k = first-layer neuron k, column i = coefficient on ODT normal u_i.
init_coeffs = np.zeros((WIDTH_1, NUM_INTERNAL_NODES), dtype=np.float64)

for node in V_NODES:        # w_2, w_3, w_4 all get -eps * v
    init_coeffs[1, node] -= EPS
    init_coeffs[2, node] -= EPS
    init_coeffs[3, node] -= EPS
init_coeffs[2, 7] -= EPS    # w_3 extra tilt toward -u_7
init_coeffs[3, 8] -= EPS    # w_4 extra tilt toward -u_8

# ---------------------------------------------------------------------------


def build_first_layer_weights(
    coeffs: np.ndarray,
    tree,
    *,
    random_scale: float = 0.0,
    seed: int = 0,
) -> np.ndarray:
    """Build W_1 from ODT-basis coefficients plus an off-ODT random component.

    The random part is projected out of the ODT span, so ``w_k . u_i`` equals
    ``coeffs[k, i]`` exactly regardless of ``random_scale``.
    """
    odt_normals = tree.w_list
    weights = np.asarray(coeffs, dtype=np.float64) @ odt_normals
    if random_scale > 0:
        rng = np.random.default_rng(seed)
        spurious = rng.standard_normal(weights.shape)
        spurious -= (spurious @ odt_normals.T) @ odt_normals
        spurious /= np.linalg.norm(spurious, axis=1, keepdims=True)
        weights = weights + random_scale * spurious
    return weights


W1_INIT = build_first_layer_weights(
    init_coeffs, tree, random_scale=INIT_RANDOM_SCALE, seed=INIT_RANDOM_SEED
)

init_view = pd.DataFrame(
    W1_INIT @ U.T,
    index=[f"w{k + 1}" for k in range(WIDTH_1)],
    columns=[f"u{i}" for i in range(NUM_INTERNAL_NODES)],
)
init_view["||w||"] = np.linalg.norm(W1_INIT, axis=1)
init_view["off-ODT norm"] = np.linalg.norm(W1_INIT - (W1_INIT @ U.T) @ U, axis=1)
init_view.round(4)

## 5. Build the model and run population gradient flow

In [ ]:
def build_frozen_sign_model(
    w1_init: np.ndarray,
    *,
    sign_patterns: np.ndarray = SIGN_PATTERNS,
    output_signs: np.ndarray = OUTPUT_SIGNS,
) -> ReLUMLP:
    """ReLU MLP with hand-set first layer and frozen sign-pattern / output layers."""
    model = ReLUMLP(input_dim=DIM, hidden_dims=[WIDTH_1, WIDTH_2], bias=False)
    with torch.no_grad():
        model.hidden_layers[0].weight.copy_(torch.as_tensor(w1_init, dtype=torch.float32))
        model.hidden_layers[1].weight.copy_(torch.as_tensor(sign_patterns, dtype=torch.float32))
        model.output_layer.weight.copy_(
            torch.as_tensor(output_signs, dtype=torch.float32).reshape(1, WIDTH_2)
        )
    return model


def first_layer_only_mask() -> dict[str, np.ndarray]:
    """Update mask that trains W_1 and freezes both downstream layers."""
    return {
        "hidden_layers.0.weight": np.ones((WIDTH_1, DIM), dtype=np.float32),
        "hidden_layers.1.weight": np.zeros((WIDTH_2, WIDTH_1), dtype=np.float32),
        "output_layer.weight": np.zeros((1, WIDTH_2), dtype=np.float32),
    }


model = build_frozen_sign_model(W1_INIT)

out = train_relu_mlp(
    model=model,
    x_train=x_train,
    y_train=y_train,
    config=ReLUTrainConfig(
        epochs=TOTAL_EPOCHS,
        lr=LR,
        batch_size=N_TRAIN,               # full batch = population gradient
        seed=TRAIN_SEED,
        device=DEVICE,
        snapshot_epochs=SNAPSHOT_EPOCHS,
        weight_decay=0.0,
        lr_scheduler_eta_min_ratio=1.0,   # constant step size (gradient flow)
        show_progress=True,
        optimizer="sgd",
        momentum=0.0,
        parameter_update_mask=first_layer_only_mask(),
    ),
)

run = {
    "master_seed": DATA_SEED,
    "seeds": {"data_seed": DATA_SEED, "train_seed": TRAIN_SEED},
    "tree": tree,
    "meta": meta,
    "data": {"x_train": x_train, "y_train": y_train, "x_eval": x_eval, "y_eval": y_eval},
    "out": out,
}
ckpts = out["checkpoint_snapshots"]
epochs_available = sorted(ckpts)
FINAL_EPOCH = epochs_available[-1]

print("checkpoints:", len(epochs_available), "final epoch:", FINAL_EPOCH)
pd.DataFrame(collect_start_end_metrics(run))

### Freeze check

Layers 2 and 3 must be bit-identical to their initial values, and the first layer's
ODT coordinates at epoch 0 must match `init_coeffs`.

In [ ]:
first_state, last_state = ckpts[epochs_available[0]], ckpts[FINAL_EPOCH]

assert torch.equal(first_state["hidden_layers.1.weight"], last_state["hidden_layers.1.weight"]), \
    "layer-2 sign patterns moved"
assert torch.equal(first_state["output_layer.weight"], last_state["output_layer.weight"]), \
    "output signs moved"
assert np.allclose(
    first_state["hidden_layers.0.weight"].numpy() @ U.T, init_coeffs, atol=1e-5
), "first-layer init does not match init_coeffs"
assert not torch.equal(
    first_state["hidden_layers.0.weight"], last_state["hidden_layers.0.weight"]
), "first layer did not move"

print("frozen layers unchanged; first layer moved; init matches init_coeffs")
print("final |W1| rows:", np.round(np.linalg.norm(last_state["hidden_layers.0.weight"].numpy(), axis=1), 4))

### Validating the lens formula

Before reading any trajectory, check that the object the theory talks about is the object
the code computes: assemble $-\nabla_{w_k}\mathcal{L}$ by hand from
residue $\times$ lens $\times$ blinder $\times\, y\,x$, with the lens taken from
`first_layer_neuron_weighted_lens`, and compare against autograd.

In [ ]:
def lens_matrix(model: ReLUMLP, x: np.ndarray, device: str = DEVICE) -> np.ndarray:
    """Lambda_k(x) for every first-layer neuron: shape (WIDTH_1, n)."""
    acts = activation_tensor_by_layer_neuron_data(model=model, x_eval=x, device=device)
    state = model.state_dict()
    return np.stack(
        [
            first_layer_neuron_weighted_lens(
                state_dict=state, activation_tensor=acts, first_layer_neuron_id=k
            )
            for k in range(WIDTH_1)
        ]
    )


_check_n = min(20_000, N_TRAIN)
_x, _y = x_train[:_check_n], y_train[:_check_n]
_model = checkpoint_model_from_state(out["model"], ckpts[FINAL_EPOCH])

_lens = lens_matrix(_model, _x)
with torch.no_grad():
    _logits = _model(torch.from_numpy(_x).float().to(DEVICE)).cpu().numpy()
_residue = 1.0 / (1.0 + np.exp(_y * _logits))            # sigma(-y * yhat)
_manual = (_residue * _y * _lens) @ _x / _check_n        # = -grad, shape (WIDTH_1, d)

_autograd = -log_loss_gradients(
    model=_model, x=_x, y=_y, device=DEVICE
)["hidden_layers.0.weight"].numpy()

print("max abs difference (manual residue*lens*blinder vs autograd):",
      float(np.abs(_manual - _autograd).max()))
print("relative:", float(np.abs(_manual - _autograd).max() / np.abs(_autograd).max()))
assert np.allclose(_manual, _autograd, atol=1e-6), "lens decomposition does not reproduce the gradient"

## 6. Parameter trajectory (paper Fig. 2, top)

One panel per first-layer neuron; each line is $w_k \cdot u_i$ over epochs. Solid lines
are internal ODT nodes (levels $0..D-2$), dashed lines the deepest-level nodes
($u_7..u_{14}$ for $D=4$), and the black dotted line is the norm of the component
orthogonal to every ODT direction — the "spurious" mass that Thm. 2 is about.

In [ ]:
def odt_coordinate_trajectories(ckpts, tree, param_key="hidden_layers.0.weight"):
    """Return (epochs, coords[E, K, N_nodes], off_odt_norm[E, K])."""
    odt_normals = tree.w_list
    epochs = np.array(sorted(ckpts), dtype=int)
    coords, residual = [], []
    for epoch in epochs:
        weights = ckpts[int(epoch)][param_key].numpy()
        projection = weights @ odt_normals.T
        coords.append(projection)
        residual.append(np.linalg.norm(weights - projection @ odt_normals, axis=1))
    return epochs, np.stack(coords), np.stack(residual)


def plot_odt_trajectories(
    epochs,
    values,
    *,
    residual=None,
    epoch_limit=None,
    ylabel=r"$w_k \cdot u_i$",
    suptitle="",
):
    """Paper-style panel per first-layer neuron, one line per ODT direction."""
    keep = slice(None) if epoch_limit is None else epochs <= epoch_limit
    ep = epochs[keep]
    num_neurons = values.shape[1]
    colors = plt.get_cmap("tab20", NUM_INTERNAL_NODES)

    fig, axes = plt.subplots(
        1, num_neurons, figsize=(4.6 * num_neurons, 4.4), sharey=True, squeeze=False
    )
    for k in range(num_neurons):
        ax = axes[0, k]
        for node_id in range(NUM_INTERNAL_NODES):
            deep = node_id in DEEP_NODES
            ax.plot(
                ep,
                values[keep, k, node_id],
                color=colors(node_id),
                ls="dashed" if deep else "solid",
                lw=1.3 if deep else 1.8,
                label=f"$u_{{{node_id}}}$",
            )
        if residual is not None:
            ax.plot(ep, residual[keep, k], color="k", ls="dotted", lw=1.6, label="off-ODT norm")
        ax.axhline(0.0, color="k", lw=0.6)
        ax.set_title(rf"$w_{{{k + 1}}}$")
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.2)
    axes[0, 0].set_ylabel(ylabel)
    axes[0, -1].legend(fontsize=7, ncol=2, loc="center left", bbox_to_anchor=(1.02, 0.5))
    if suptitle:
        fig.suptitle(suptitle, y=1.03)
    fig.tight_layout()
    return fig, axes


traj_epochs, traj_coords, traj_residual = odt_coordinate_trajectories(ckpts, tree)

TRAJ_EPOCH_LIMIT = None   # set e.g. 400 to match the paper's 400-epoch figure

fig_param, _ = plot_odt_trajectories(
    traj_epochs,
    traj_coords,
    residual=traj_residual,
    epoch_limit=TRAJ_EPOCH_LIMIT,
    ylabel=r"$w_k \cdot u_i$",
    suptitle="Parameter trajectory: first-layer weights in the ODT basis",
)
plt.show()

## 7. Gradient trajectory (paper Fig. 2, bottom)

The same panels for $-\nabla_{w_k}\mathcal{L}\cdot u_i$, i.e. the direction the flow is
currently pushing each neuron. This is where the second-order story is legible in the
DLGN run: the push on $w_1$ toward $-u_0$ builds up *after* the other neurons have moved
along the deepest-level directions, then the push on $w_2$ toward $-u_1$ follows.

In [ ]:
grad_checkpoint_epochs = [
    epochs_available[i]
    for i in np.unique(
        np.linspace(0, len(epochs_available) - 1, GRAD_TRAJ_MAX_CHECKPOINTS).astype(int)
    )
]

grad_coords = []
for epoch in grad_checkpoint_epochs:
    model_at_epoch = checkpoint_model_from_state(out["model"], ckpts[epoch])
    grads = log_loss_gradients(model=model_at_epoch, x=x_train, y=y_train, device=DEVICE)
    neg_grad = -grads["hidden_layers.0.weight"].numpy()
    grad_coords.append(neg_grad @ U.T)

grad_epochs = np.array(grad_checkpoint_epochs, dtype=int)
grad_coords = np.stack(grad_coords)

fig_grad, _ = plot_odt_trajectories(
    grad_epochs,
    grad_coords,
    epoch_limit=TRAJ_EPOCH_LIMIT,
    ylabel=r"$-\nabla_{w_k}\mathcal{L} \cdot u_i$",
    suptitle="Gradient trajectory: population gradient in the ODT basis",
)
plt.show()

## 8. Where each neuron ended up

The $\upsilon$-tuple $\upsilon_k = \arg\max_i |w_k \cdot u_i|$ is the paper's compact
solution label (the DLGN trajectory gave $[0,1,7,8]$).

In [ ]:
final_weights = ckpts[FINAL_EPOCH]["hidden_layers.0.weight"].numpy()
final_coords = final_weights @ U.T
final_norms = np.linalg.norm(final_weights, axis=1)

upsilon = np.abs(final_coords).argmax(axis=1)
signed = final_coords[np.arange(WIDTH_1), upsilon]

summary = pd.DataFrame(
    {
        "neuron": [f"w{k + 1}" for k in range(WIDTH_1)],
        "upsilon_node": upsilon,
        "odt_level": NODE_LEVEL[upsilon],
        "signed_coord": signed,
        "cos": signed / np.maximum(final_norms, 1e-12),
        "norm": final_norms,
        "off_odt_norm": np.linalg.norm(final_weights - final_coords @ U, axis=1),
    }
)
summary["aligned"] = summary["cos"].abs() >= COS_THRESHOLD
print("upsilon tuple:", [int(v) for v in upsilon])
print("signed solution:",
      [f"{'+' if c > 0 else '-'}u{n}" for c, n in zip(signed, upsilon)])
summary.round(4)

In [ ]:
capture_rows = []
abs_cos_traj = np.abs(traj_coords) / np.maximum(
    np.linalg.norm(
        np.stack([ckpts[int(e)]["hidden_layers.0.weight"].numpy() for e in traj_epochs]),
        axis=2,
    )[:, :, None],
    1e-12,
)
for node_id in range(NUM_INTERNAL_NODES):
    hit = np.nonzero((abs_cos_traj[:, :, node_id] >= COS_THRESHOLD).any(axis=1))[0]
    capture_rows.append(
        {
            "odt_node_id": node_id,
            "odt_level": int(NODE_LEVEL[node_id]),
            "first_epoch": float(traj_epochs[hit[0]]) if hit.size else np.nan,
        }
    )
capture_df = pd.DataFrame(capture_rows)

fig_tree, _ = plot_complete_binary_tree_values(
    capture_df["first_epoch"].to_numpy(dtype=float),
    title=f"First epoch a first-layer neuron reaches |cos| > {COS_THRESHOLD}",
    suptitle="",
)
plt.show()
capture_df

## 9. The lens, unpacked by sign pattern

$\Lambda_k = \sum_s a_s s_k \mathbb{1}[z_s>0]$ is a sum over the $2^D$ frozen units, so it
can be read off unit by unit: which sign patterns are active, on which ODT leaf regions,
and how much each contributes to the lens of each neuron. This is the ReLU replacement
for "the product of the other gates".

In [ ]:
final_model = checkpoint_model_from_state(out["model"], ckpts[FINAL_EPOCH])
final_acts = activation_tensor_by_layer_neuron_data(
    model=final_model, x_eval=x_train, device=DEVICE
)
unit_active = final_acts[1, :WIDTH_2]                  # (2^D, n) layer-2 activity

pattern_df = frozen_layers_df.copy()
pattern_df["active_fraction"] = unit_active.mean(axis=1)
for k in range(WIDTH_1):
    pattern_df[f"lens_share_w{k + 1}"] = (
        OUTPUT_SIGNS * SIGN_PATTERNS[:, k] * unit_active.mean(axis=1)
    )
pattern_df.round(4)

In [ ]:
final_lens = lens_matrix(final_model, x_train)

lens_by_leaf = np.stack(
    [
        np.bincount(leaf_offsets, weights=final_lens[k], minlength=NUM_LEAVES)
        / np.maximum(np.bincount(leaf_offsets, minlength=NUM_LEAVES), 1)
        for k in range(WIDTH_1)
    ]
)

fig, ax = plt.subplots(figsize=(10, 3.4))
im = ax.imshow(lens_by_leaf, aspect="auto", cmap="coolwarm",
               vmin=-np.abs(lens_by_leaf).max(), vmax=np.abs(lens_by_leaf).max())
ax.set_yticks(range(WIDTH_1), [f"$w_{{{k + 1}}}$" for k in range(WIDTH_1)])
ax.set_xticks(range(NUM_LEAVES), [str(NUM_INTERNAL_NODES + j) for j in range(NUM_LEAVES)])
ax.set_xlabel("ODT leaf node")
ax.set_title(r"mean lens $\Lambda_k(x)$ per ODT leaf region (final epoch)")
fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()

pd.DataFrame(
    lens_by_leaf,
    index=[f"w{k + 1}" for k in range(WIDTH_1)],
    columns=[f"leaf{NUM_INTERNAL_NODES + j}" for j in range(NUM_LEAVES)],
).round(4)

## 10. Hierarchy-seeking probe (Theorem 1 analog)

The theorem's quantity, measured directly:

$$\lim_{\epsilon\to0}\frac{1}{\epsilon}\Big[-\nabla_{w_k}\mathcal{L}(W+\epsilon P_\ell h)
+\nabla_{w_k}\mathcal{L}(W)\Big]\cdot z ,$$

for $h = u_j$ with $j$ a deepest-level node, $z = s\,u_i$ with $i$ an ancestor of $j$, and
$s=+1$ if $j$ is in $i$'s **left** subtree, $-1$ otherwise. The DLGN theorem says this is
$>0$ for $k\neq\ell$. Here it is estimated by a central finite difference on the
population gradient, evaluated independently on two disjoint halves of the data so the
estimate's own reliability is visible: with `PROBE_EPS` too small the difference is
swamped by sampling noise, too large and it picks up curvature beyond the derivative.
At $d=100$, $n=10^5$ per half, `PROBE_EPS=1e-2` was split-half sign-stable on 95% of
entries, `1e-3` on 84%, and `1e-4` on 60%.

Left/right convention follows `odt.py`: a point goes **right** when $x\cdot u_i > 0$, so
node $2i+1$ is the left child.

In [ ]:
def ancestors_with_side(node_id: int) -> list[tuple[int, int]]:
    """[(ancestor_id, +1 if node is in its left subtree else -1), ...], root first."""
    out_pairs = []
    node = node_id
    while node > 0:
        parent = (node - 1) // 2
        out_pairs.append((parent, 1 if node == 2 * parent + 1 else -1))
        node = parent
    return out_pairs[::-1]


def second_order_probe(
    state_dict: dict[str, torch.Tensor],
    *,
    perturb_neuron: int,
    perturb_dir: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    eps: float = PROBE_EPS,
    device: str = DEVICE,
) -> np.ndarray:
    """Central-difference d/d(eps) of the negative first-layer gradient: (WIDTH_1, d)."""
    weights = state_dict["hidden_layers.0.weight"].numpy().copy()
    grads = []
    for sign in (+1.0, -1.0):
        perturbed = weights.copy()
        perturbed[perturb_neuron] = perturbed[perturb_neuron] + sign * eps * perturb_dir
        probe_state = {k: v.clone() for k, v in state_dict.items()}
        probe_state["hidden_layers.0.weight"] = torch.as_tensor(perturbed, dtype=torch.float32)
        probe_model = checkpoint_model_from_state(out["model"], probe_state)
        grads.append(
            -log_loss_gradients(model=probe_model, x=x, y=y, device=device)[
                "hidden_layers.0.weight"
            ].numpy()
        )
    return (grads[0] - grads[1]) / (2.0 * eps)


PROBE_EPOCH = epochs_available[0]        # at the hand-picked initialisation
probe_state = ckpts[PROBE_EPOCH]

assert 2 * PROBE_N <= N_TRAIN, "need two disjoint probe halves; lower PROBE_N"
x_probe, y_probe = x_train[:PROBE_N], y_train[:PROBE_N]
x_probe_b, y_probe_b = x_train[PROBE_N : 2 * PROBE_N], y_train[PROBE_N : 2 * PROBE_N]


def hierarchy_seeking_table(state_dict, *, output_signs=None) -> pd.DataFrame:
    """Theorem-1 responses on two disjoint data halves, for every (ell, j, k, ancestor)."""
    state_dict = {key: value.clone() for key, value in state_dict.items()}
    if output_signs is not None:
        state_dict["output_layer.weight"] = torch.as_tensor(
            np.asarray(output_signs, dtype=np.float64), dtype=torch.float32
        ).reshape(1, WIDTH_2)

    rows = []
    for ell in range(WIDTH_1):
        for j in DEEP_NODES:
            resp_a = second_order_probe(
                state_dict, perturb_neuron=ell, perturb_dir=U[j], x=x_probe, y=y_probe
            )
            resp_b = second_order_probe(
                state_dict, perturb_neuron=ell, perturb_dir=U[j], x=x_probe_b, y=y_probe_b
            )
            for k in range(WIDTH_1):
                if k == ell:
                    continue
                for ancestor, side in ancestors_with_side(j):
                    z = side * U[ancestor]
                    value_a, value_b = float(resp_a[k] @ z), float(resp_b[k] @ z)
                    rows.append(
                        {
                            "perturbed_neuron": ell,
                            "h_node": j,
                            "observed_neuron": k,
                            "ancestor_node": ancestor,
                            "ancestor_level": int(NODE_LEVEL[ancestor]),
                            "predicted_sign": side,
                            "response": 0.5 * (value_a + value_b),
                            "response_half_a": value_a,
                            "response_half_b": value_b,
                            "sign_stable": (value_a > 0) == (value_b > 0),
                        }
                    )
    table = pd.DataFrame(rows)
    table["matches_theorem"] = table["response"] > 0
    return table


hs_df = hierarchy_seeking_table(probe_state)

stability = hs_df["sign_stable"].mean()
correlation = np.corrcoef(hs_df["response_half_a"], hs_df["response_half_b"])[0, 1]
print(f"probe epoch {PROBE_EPOCH}, n={PROBE_N} per half, eps={PROBE_EPS}")
print(f"split-half sign stability: {stability:.3f}  (correlation {correlation:.3f})")
if stability < 0.9:
    print("  WARNING: the finite difference is noise-dominated here. Raise PROBE_EPS "
          "or PROBE_N before reading the agreement number.")
print(f"Theorem-1 sign agreement: {hs_df['matches_theorem'].mean():.3f} "
      f"({int(hs_df['matches_theorem'].sum())}/{len(hs_df)})")
hs_df.groupby("ancestor_level")[["response", "matches_theorem"]].agg(
    ["mean", "count"]
).round(5)

In [ ]:
pivot = hs_df.pivot_table(
    index=["perturbed_neuron", "h_node"],
    columns=["observed_neuron", "ancestor_node"],
    values="response",
)

fig, ax = plt.subplots(figsize=(13, 6))
limit = np.nanmax(np.abs(pivot.to_numpy()))
im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-limit, vmax=limit)
ax.set_yticks(range(len(pivot.index)),
              [f"$w_{{{ell + 1}}}$ + $u_{{{j}}}$" for ell, j in pivot.index], fontsize=8)
ax.set_xticks(range(len(pivot.columns)),
              [f"$w_{{{k + 1}}}\\cdot s u_{{{i}}}$" for k, i in pivot.columns],
              rotation=90, fontsize=8)
ax.set_title("Hierarchy-seeking response (red > 0 agrees with Theorem 1)")
fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()

### Swapped roles

Theorem 1 also asserts the inequality with $h$ and $z$ exchanged: perturbing a neuron
along an ancestor direction should increase another neuron's push along the
(sign-corrected) deepest-level direction. That is the other half of the feedback loop.

In [ ]:
swap_rows = []
for ell in range(WIDTH_1):
    for j in DEEP_NODES:
        for ancestor, side in ancestors_with_side(j):
            response = second_order_probe(
                probe_state,
                perturb_neuron=ell,
                perturb_dir=side * U[ancestor],
                x=x_probe,
                y=y_probe,
            )
            for k in range(WIDTH_1):
                if k == ell:
                    continue
                swap_rows.append(
                    {
                        "perturbed_neuron": ell,
                        "h_node": ancestor,
                        "observed_neuron": k,
                        "z_node": j,
                        "ancestor_level": int(NODE_LEVEL[ancestor]),
                        "response": float(response[k] @ U[j]),
                    }
                )

swap_df = pd.DataFrame(swap_rows)
swap_df["matches_theorem"] = swap_df["response"] > 0
print(f"swapped-role sign agreement: {swap_df['matches_theorem'].mean():.3f} "
      f"({int(swap_df['matches_theorem'].sum())}/{len(swap_df)})")
swap_df.groupby("ancestor_level")[["response", "matches_theorem"]].agg(
    ["mean", "count"]
).round(5)

### Does the frozen output vector decide whether the effect appears?

$\Lambda_k = \sum_s a_s s_k \mathbb{1}[z_s>0]$ is a **signed** sum, so the frozen $a$
chooses which conjunctions the lens rewards — unlike the DLGN, where the theorem is
proved with a single fixed $a>0$. `all_positive` is that case; `random` is the
specification used above. This cell repeats the full Theorem-1 grid per preset, so it
costs one probe grid per entry.

A preliminary run (d=100, $10^5$ per half, `PROBE_EPS=1e-2`, paper init) gave agreement
`all_positive` 0.93, `majority` 0.94, `parity` 0.76, and for random vectors
0.37 / 0.70 / 0.50 at `OUTPUT_SIGN_SEED` 178 / 0 / 42. With a random $\pm1$ output the
hierarchy-seeking sign structure is therefore seed-dependent and can sit *below* chance,
while the direct analog of the paper's $a>0$ reproduces it strongly. Worth settling
before reading much into a single random draw.

In [ ]:
OUTPUT_SIGN_SWEEP = ["random", "all_positive", "majority", "parity"]

sweep_rows = []
for preset_name in OUTPUT_SIGN_SWEEP:
    table = hierarchy_seeking_table(
        probe_state, output_signs=OUTPUT_SIGN_PRESETS[preset_name]
    )
    sweep_rows.append(
        {
            "output_signs": preset_name,
            "thm1_agreement": table["matches_theorem"].mean(),
            "median_response": table["response"].median(),
            "split_half_stability": table["sign_stable"].mean(),
        }
    )

pd.DataFrame(sweep_rows).round(4)

## 11. Self-opposing probe (Theorem 2 analog)

For $h$ orthogonal to **every** ODT direction, the DLGN theorem says perturbing $w_\ell$
along $h$ increases the push on other neurons along $-h$: neurons amplify each other's
spurious components. Averaged over random spurious directions, a positive response here
is the mechanism behind the off-ODT mass tracked as the black dotted line in Sec. 6.

In [ ]:
SELF_OPPOSING_DIRS = 20

rng = np.random.default_rng(INIT_RANDOM_SEED + 1)
so_rows = []
for trial in range(SELF_OPPOSING_DIRS):
    h = rng.standard_normal(DIM)
    h -= (h @ U.T) @ U                       # orthogonal to every ODT normal
    h /= np.linalg.norm(h)
    ell = trial % WIDTH_1
    response = second_order_probe(
        probe_state, perturb_neuron=ell, perturb_dir=h, x=x_probe, y=y_probe
    )
    for k in range(WIDTH_1):
        if k == ell:
            continue
        so_rows.append(
            {
                "trial": trial,
                "perturbed_neuron": ell,
                "observed_neuron": k,
                "response_along_minus_h": float(response[k] @ (-h)),
            }
        )

so_df = pd.DataFrame(so_rows)
so_df["matches_theorem"] = so_df["response_along_minus_h"] > 0
print(f"Theorem-2 sign agreement: {so_df['matches_theorem'].mean():.3f} "
      f"({int(so_df['matches_theorem'].sum())}/{len(so_df)})")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(so_df["response_along_minus_h"], bins=25, color="tab:purple", alpha=0.85)
ax.axvline(0.0, color="k", lw=1)
ax.set_xlabel(r"response along $-h$  (positive agrees with Theorem 2)")
ax.set_ylabel("count")
ax.set_title("Self-opposing lens probe, spurious directions")
fig.tight_layout()
plt.show()

so_df.describe().round(6)

## 12. Sweeping hand-picked initialisations

Edit `INIT_VARIANTS` to compare constructions: each entry is a
`(name, coefficient matrix)` pair, trained with identical data and frozen layers. Use it
to check that different perturbations steer the flow to different $\upsilon$-tuples, the
ReLU version of the paper's claim that any canonical solution is reachable by a suitable
small perturbation.

In [ ]:
def coeffs_paper(eps: float = EPS) -> np.ndarray:
    """Paper Sec. 3.3 perturbation: w1=0, w2=-eps v, w3=-eps v -eps u7, w4=-eps v -eps u8."""
    coeffs = np.zeros((WIDTH_1, NUM_INTERNAL_NODES))
    for node in V_NODES:
        coeffs[1:, node] -= eps
    coeffs[2, 7] -= eps
    coeffs[3, 8] -= eps
    return coeffs


def coeffs_saddle(eps: float = EPS) -> np.ndarray:
    """Symmetric saddle: every neuron at c * sum_j -u_j over deepest-level nodes."""
    coeffs = np.zeros((WIDTH_1, NUM_INTERNAL_NODES))
    coeffs[:, DEEP_NODES] -= eps
    return coeffs


def coeffs_leaf_pair(leaf_a: int = 9, leaf_b: int = 10, eps: float = EPS) -> np.ndarray:
    """Saddle plus tilts toward a different adjacent leaf pair."""
    coeffs = coeffs_saddle(eps)
    coeffs[0] = 0.0
    coeffs[2, leaf_a] -= eps
    coeffs[3, leaf_b] -= eps
    return coeffs


INIT_VARIANTS = {
    "paper (7, 8)": coeffs_paper(),
    "saddle only": coeffs_saddle(),
    "leaf pair (9, 10)": coeffs_leaf_pair(),
}

variant_rows = []
variant_runs = {}
for name, coeffs in INIT_VARIANTS.items():
    variant_model = build_frozen_sign_model(
        build_first_layer_weights(
            coeffs, tree, random_scale=INIT_RANDOM_SCALE, seed=INIT_RANDOM_SEED
        )
    )
    variant_out = train_relu_mlp(
        model=variant_model,
        x_train=x_train,
        y_train=y_train,
        config=ReLUTrainConfig(
            epochs=TOTAL_EPOCHS,
            lr=LR,
            batch_size=N_TRAIN,
            seed=TRAIN_SEED,
            device=DEVICE,
            snapshot_epochs=(0, TOTAL_EPOCHS),
            weight_decay=0.0,
            lr_scheduler_eta_min_ratio=1.0,
            show_progress=False,
            optimizer="sgd",
            momentum=0.0,
            parameter_update_mask=first_layer_only_mask(),
        ),
    )
    variant_runs[name] = variant_out
    variant_weights = variant_out["checkpoint_snapshots"][TOTAL_EPOCHS][
        "hidden_layers.0.weight"
    ].numpy()
    variant_coords = variant_weights @ U.T
    variant_upsilon = np.abs(variant_coords).argmax(axis=1)
    variant_signed = variant_coords[np.arange(WIDTH_1), variant_upsilon]
    variant_rows.append(
        {
            "init": name,
            "upsilon": tuple(int(v) for v in variant_upsilon),
            "signed": tuple(
                f"{'+' if c > 0 else '-'}u{n}" for c, n in zip(variant_signed, variant_upsilon)
            ),
            "final_log_loss": variant_out["epoch_losses"][-1],
            "max_abs_cos": float(
                np.max(np.abs(variant_signed) / np.linalg.norm(variant_weights, axis=1))
            ),
        }
    )

pd.DataFrame(variant_rows)

## 13. Notes and open threads

- **Reading the trajectories.** The DLGN narrative to check against: an early phase
  moving to the symmetric saddle, then a push on $w_1$ along $-u_0$ that builds only
  after the other neurons have committed to deepest-level directions, then $w_2$ along
  $-u_1$, with a brief $+u_0$ backlash on the other neurons (Thm. 2) while $w_1$'s push
  is strongest.
- **What differs from DLGN by construction.** The lens is a *signed sum* over $2^D$
  frozen units rather than a product, so it can change sign as a function of $x$, and a
  random frozen $a_s$ can make some sign patterns actively unhelpful. The sweep in
  Sec. 10 measures how much this matters, and in a preliminary run it mattered a lot:
  Theorem-1 agreement ran 0.93-0.94 for $a \ge 0$ / majority signs but 0.37-0.70 across
  three random draws. If the goal is the closest possible analog of the paper's theorem,
  `OUTPUT_SIGN_CHOICE = "all_positive"` is the setting to use; if the goal is the
  random-$a$ model as specified, the seed dependence is itself the finding to chase.
- **Probe location.** Sec. 10-11 probe the initialisation, where the theorems'
  orthogonality assumptions hold. Re-running with `PROBE_EPOCH` set to a mid-training
  checkpoint asks a different and also interesting question: whether the sign structure
  survives once the weights carry real ODT mass.
- **Still to do:** vary $\epsilon$ and `INIT_RANDOM_SCALE` to locate the boundary between
  perturbation-driven and noise-driven outcomes; add the $a_s$ hand-designs that make the
  net able to express a single ODT path; and if the trajectories reproduce the DLGN
  staging, move `odt_coordinate_trajectories`, `second_order_probe`, and
  `ancestors_with_side` into `src/layer_lenses/relu_analysis.py`.